Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [4]:
words = open("names.txt", "r").read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [5]:
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0

itos = {i:s for s,i in stoi.items()}


In [6]:
block_size = 7

# prepare dataset 
def build_dataset(words):
    #sliding window
    X, Y = [], []
    
    for w in words:
        cxt = [0] * block_size 
    
        for ch in w + '.':
            ix = stoi[ch]
            X.append(cxt)
            Y.append(ix)
            # print(''.join(itos[char] for char in cxt), "-->", itos[ix])
            cxt = cxt[1:] + [ix] 
    
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

# --- 

import random 
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words)) 

Xtr, Ytr = build_dataset(words[:n1])
Xtest, Ytest = build_dataset(words[n1:n2])
Xeval, Yeval = build_dataset(words[n2:])


torch.Size([182625, 7]) torch.Size([182625])
torch.Size([22655, 7]) torch.Size([22655])
torch.Size([22866, 7]) torch.Size([22866])


In [7]:
embed_dim = 30
num_neuron = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, embed_dim))

# take 100 neurons for the first layer as an example 
W1 = torch.randn((block_size*embed_dim, num_neuron), generator = g)
b1 = torch.randn(num_neuron)

W2 = torch.randn((num_neuron,27), generator = g)
b2 = torch.randn(27, generator = g)

In [8]:
### notessss 

# C[5]
# alternative indexing through matrix multiplication
# F.one_hot(torch.tensor(5), num_classes = 27).float() @ C

# torch.cat((emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]), 1) -> not scalable code 
# torch.cat(torch.unbind(emb, 1), 1) -> manual transformation 

In [9]:
param = [C, W1, b1, W2, b2]

for p in param:
    p.requires_grad = True

sum(p.nelement() for p in param)

48437

In [ ]:
### training 
lr = 0.1 # around 10**(-1)
epoch = 200000
batch_size = 128

step_i = []
loss_i = []

lr = torch.linspace(0.1, 10**(-8), epoch)



for i in range(epoch):
    # minibatch 
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    
    # forward pass 
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, block_size*embed_dim) @ W1 + b1) # (batch_size, 100)
    logits = h @ W2 + b2 # (batch_size, 27)
    loss = F.cross_entropy(logits, Ytr[ix])

    step_i.append(i)
    loss_i.append(loss.item())
    
    # backward pass 
    for p in param:
        p.grad = None
    
    loss.backward()
    
    for p in param:
        p.data -= lr[i] * p.grad



In [ ]:
plt.plot(step_i, loss_i)
print(min(loss_i))

In [ ]:
### against training batch 
emb = C[Xtr]
h = torch.tanh(emb.view(-1, block_size* embed_dim) @ W1 + b1) # (batch_size, 100)
logits = h @ W2 + b2 # (batch_size, 27)
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
### eval against test batch 
emb = C[Xtest]
h = torch.tanh(emb.view(-1, block_size*embed_dim) @ W1 + b1) # (batch_size, 100)
logits = h @ W2 + b2 # (batch_size, 27)
loss = F.cross_entropy(logits, Ytest)
print(loss) 

In [ ]:
context = [0] * block_size
C[torch.tensor([context])].shape

In [ ]:
### sampling 
sample_g = torch.Generator().manual_seed(2147483647+10)


for _ in range(20):
    context = [0] * block_size  
    output = []
    while True:
        emb = C[context] # block_size * embed_dim 
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        prob = F.softmax(logits, dim = 1)
        ix = torch.multinomial(prob, num_samples = 1, generator = sample_g).item()
        output.append(ix)
        
        context = context[1:] + [ix]
        if ix == 0:
            break
    print(''.join(itos[i] for i in output))
    
    

In [ ]:
h.shape